# Direct Random Forest: устойчивость признаков и параметров

Самодостаточный ноутбук Kaggle: NVIDIA GPU, cuML и CuPy. По умолчанию direct и все восемь горизонтов. Обучение запускает пользователь.

## Протокол и временные границы

Сравниваются три кандидата: baseline, thin_offsets и compact_with_additions. Первый сохраняет исходный набор, второй прореживает несезонные смещения (центр и −2 вместо пяти значений), третий сокращает окна, календарь, сетевые смещения и флаги доступности, удаляет selected_status6_pair_count и добавляет время старта, profile28, level24, аномалии, тренды, изменчивость и сезонный разброс.

Сначала все кандидаты обучаются с BASE_PARAMS_BY_HORIZON: лучшие параметры каждого горизонта из artifacts/rf_direct/model_params/best_hN.json встроены в первую ячейку. Для данного горизонта параметры одинаковы у всех кандидатов. Старые tuning-score не используются: конфигурации заново оцениваются на шести окнах. Train использует пять стартов 09:30–11:30, tuning — 10:30. Шесть недельных tuning-окон: 30 августа — 5 сентября, 6–12 сентября, 13–19 сентября, 20–26 сентября, 27 сентября — 3 октября, 4–10 октября 2025 года. Последние три окна — дополнительная проверка переноса во времени. Перед каждым окном оставлена неделя val-зазора. Корреляционные лаги и медианы обучаются только на train соответствующего окна. Новые признаки используют данные не позднее cutoff.

Общий набор выбирается по средней метрике шести окон; в каждом окне сначала объединяются ошибки по всем включённым горизонтам и маршрутам. Отдельно выводятся результаты и изменения к baseline для каждого горизонта и окна, число улучшений, разброс и худшее окно. Небольшое среднее улучшение не доказывает устойчивость при одном seed.

Затем для выбранного набора запускается Optuna отдельно на каждом горизонте: 20 trials, включая исходные параметры соответствующего горизонта первым trial. Диапазоны: 150/300 деревьев, глубина [6,8,10,12,16], min_samples_leaf 8–256 (log), max_features 0.3–0.9, max_samples 0.5–1. Критерий — среднее WAPE + |Relative Bias| по шести окнам. Глубина 16 и листья от 8 сохранены: поиск проверяет строгие и гибкие модели на новых tuning-окнах, без запрета конфигурации по просмотренному test. После поиска таблица base_vs_optuna сравнивает исходные и лучшие параметры, WAPE, signed Bias и |Bias| на каждом окне. Первый trial входит в общий бюджет. Подбор не должен проиграть собственному первому trial по средней tuning-метрике; улучшение на test не гарантируется.

Все метки подбора строго раньше 11 октября. После фиксации признаков и параметров модель обучается на всей пригодной истории до 18 октября, включая train и val, и оценивается на test 18–31 октября. Во время поиска показываются train/tuning; после фиксации — train/test. Независимая оценка получена до какого-либо переобучения на всей истории. Переобучения на всей истории, importance и submission нет. Test уже просмотрен и использовался в проекте: это контрольная оценка переноса, а не новый слепой test. Он не участвует в автоматическом отборе.

## Бюджет, кэш и продолжение

По умолчанию: 3 набора × 8 горизонтов × 6 окон = 144 обучения сравнения; 20 × 8 × 6 = 960 обучений Optuna; 8 для test. Всего 1112 обучений. Для пилота можно изменить HORIZONS на [1,4,6] и N_TRIALS на 10 до запуска. Число обучений и время отражаются в отчётах. n_streams=4, seed=42, битовая воспроизводимость GPU не гарантируется. PREDICTION_BATCH=32768; уменьшите при нехватке памяти.

Новая папка результатов: artifacts/random_forest_feature_experiments_rf_direct_seed_v4. Для изменённых временных границ используется отдельный кэш rf_experiment_features_robust_v3. Старые результаты автоматически не смешиваются с новым протоколом. Подготовленные матрицы хранятся на диске, train-only медианы общие для кандидатов. Повторный Run All продолжает завершённые сравнения, trials и оценку; прерванное обучение повторяется. Запускайте один экземпляр. Для другой Kaggle-сессии сохраните artifacts как dataset и укажите RESUME_FROM='/kaggle/input/.../artifacts'. При изменении конфигурации выберите новую RUN_DIR.

## Артефакты

feature_experiments_summary.csv — сравнение наборов. feature_experiments_by_fold.csv и feature_experiments_by_horizon.csv — детализация. feature_stability_by_horizon.csv — устойчивость по окнам и горизонтам; feature_deltas_by_window.csv — изменения к baseline. В direct/docs: base_vs_optuna_by_fold.csv, base_vs_optuna_summary.csv, trials_hN.csv, train/test метрики по горизонтам и в целом. В каждой таблице сохраняются WAPE и Bias отдельно, чтобы видеть взаимную компенсацию ошибок. feature_selection_frozen.json и all_parameters_frozen.json фиксируют выбор до test.

Model bundle содержит model, preprocess полного набора, metadata.selected_features и metadata.selected_indices: после preprocess.transform необходимо отобрать selected_indices перед model.predict. Модели оценки и прогнозы сохраняются по горизонтам. reports.zip содержит отчёты.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import gc
import os
import importlib.util
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
ALGORITHM = 'random_forest'
RESIDUAL = False
DEVICE = 'cuda' if ALGORITHM in ('xgb', 'catboost', 'random_forest') else 'cpu'
N_TRIALS = 3
N_STREAMS = 4
RF_TREES = [150, 300]
RF_DEPTHS = [6, 8, 10, 12, 16]
RESUME_FROM = None
N_JOBS = min(4, os.cpu_count() or 1)
SEED = 42
PREDICTION_BATCH = 32768
RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
required_modules = ['sklearn', 'optuna', 'joblib', 'psutil']
required_modules += [{'xgb': 'xgboost', 'catboost': 'catboost', 'lightgbm': 'lightgbm', 'prophet': 'prophet'}.get(ALGORITHM, 'sklearn')]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Установите недостающие пакеты до расчёта признаков: {sorted(set(missing))}')
if ALGORITHM == 'random_forest':
    if DEVICE != 'cuda':
        raise ValueError('Для RF выберите DEVICE=cuda')
    try:
        import cupy as cp
        from cuml.ensemble import RandomForestRegressor
        cp.zeros(1, dtype=cp.float32)
        cp.cuda.get_current_stream().synchronize()
    except (ImportError, OSError, RuntimeError) as exc:
        raise RuntimeError('Включите GPU в Kaggle и установите cuML и CuPy для CUDA этой сессии.') from exc
if N_TRIALS < 1 or PREDICTION_BATCH < 1:
    raise ValueError('Число trials и размер блока должны быть положительными')
DATA = Path('/kaggle/input/datasets/moonshyxd/case-rwb')
TRAIN_FILE = 'train_solo_track.parquet'
TRAIN_PATH = DATA / TRAIN_FILE
if not TRAIN_PATH.is_file() and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    if len(train_matches) != 1:
        raise ValueError('Укажите TRAIN_PATH вручную')
    TRAIN_PATH = train_matches[0]
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MODEL_NAME = 'random_forest_feature_experiments'
RUN_DIR = BASE / 'artifacts' / f'{MODEL_NAME}_rf_direct_seed_v4'
OUT = BASE / 'artifacts' / 'rf_experiment_features_robust_v3'
MODEL_DIR = RUN_DIR / 'models'
POSTPROCESS_DIR = RUN_DIR / 'postprocessing'
DOCS_DIR = RUN_DIR / 'docs'
PARAMS_DIR = RUN_DIR / 'model_params'
if RESUME_FROM is not None:
    import shutil
    source = Path(RESUME_FROM)
    if not source.is_dir():
        raise ValueError('RESUME_FROM должен указывать на сохранённую папку artifacts')
    for folder_name in (RUN_DIR.name, OUT.name):
        source_folder = source / folder_name
        destination = BASE / 'artifacts' / folder_name
        if source_folder.exists() and not destination.exists():
            shutil.copytree(source_folder, destination)
for directory in (OUT, MODEL_DIR, POSTPROCESS_DIR, DOCS_DIR, PARAMS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = 'five_starts_windows_v4_stage_safe_hpo'

VARIANTS = ['direct']
HORIZONS = list(range(1, 9))
EXPERIMENT_NAMES = ['baseline', 'thin_offsets', 'compact_with_additions']

BASE_PARAMS_BY_HORIZON = {1: {'n_estimators': 150, 'max_depth': 10, 'min_samples_leaf': 32, 'max_features': 0.6, 'max_samples': 0.8}, 2: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 24, 'max_features': 0.3406585285177396, 'max_samples': 0.9330880728874675}, 3: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 24, 'max_features': 0.3406585285177396, 'max_samples': 0.9330880728874675}, 4: {'n_estimators': 150, 'max_depth': 10, 'min_samples_leaf': 32, 'max_features': 0.6, 'max_samples': 0.8}, 5: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}, 6: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 23, 'max_features': 0.5045012539746527, 'max_samples': 0.6831809216468459}, 7: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}, 8: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}}
BASE_PARAMS_SOURCE = 'artifacts/rf_direct/model_params/best_hN.json'

def base_params_for_horizon(h):
    return dict(BASE_PARAMS_BY_HORIZON[h])

FEATURE_SOURCE_HASH = '249225566597229fb8df8820a96086b9d681c34ea738eff14b63457ec3a14879'
MODEL_SOURCE_HASH = 'a1630558e06ed129edf63532037af1ad15a9f3e5e88050898f3ef63c6a9752de'

import hashlib
import importlib.metadata

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def verify_contract(path, contract):
    if path.exists():
        if json.loads(path.read_text()) != contract:
            raise ValueError(f'Настройки или данные изменились: {path}. Используйте новую папку результатов/кэша.')
    else:
        temporary = path.with_suffix('.partial')
        temporary.write_text(json.dumps(contract, ensure_ascii=False, indent=2))
        temporary.replace(path)

FEATURE_CONTRACT = dict(source=FEATURE_SOURCE_HASH, data=file_digest(TRAIN_PATH), starts=list(TRAIN_MINUTES),
                        offsets=list(WINDOW_OFFSETS), lookback=LAG_MAX_LOOKBACK_HOURS,
                        min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
verify_contract(OUT / 'contract.json', FEATURE_CONTRACT)
PREPARED = OUT / ('prepared_' + importlib.metadata.version('scikit-learn') + '_' + MODEL_SOURCE_HASH[:16])
PREPARED.mkdir(parents=True, exist_ok=True)
RUN_CONTRACT = dict(features=FEATURE_CONTRACT, source=MODEL_SOURCE_HASH, variants=VARIANTS, horizons=HORIZONS, experiments=EXPERIMENT_NAMES, base_params=BASE_PARAMS_BY_HORIZON,
                    n_trials=N_TRIALS, seed=SEED, n_streams=N_STREAMS, trees=RF_TREES, depths=RF_DEPTHS,
                    versions={name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'optuna')},
                    cuml=__import__('cuml').__version__)
verify_contract(RUN_DIR / 'contract.json', RUN_CONTRACT)


## Данные и временное разбиение

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
end = times[-1]
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
selection_end = pd.Timestamp('2025-10-11')
if len(dates) < 14 or dates[-14].normalize() < selection_end or times[-1] < selection_end:
    raise ValueError('Недостаточно истории для отдельного test после подбора')
holdout_start = dates[-14]
selection_dates = dates[dates + 8 * STEP < selection_end]
if len(selection_dates) <= 49:
    raise ValueError('До 20 сентября недостаточно истории для train и трёх недель подбора')
folds = []
for start in range(len(selection_dates) - 42, len(selection_dates), 7):
    folds.append(dict(train=selection_dates[:start-7], val=selection_dates[start-7:start],
                      tuning=selection_dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))

## Общие признаки пяти стартов

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

## Корреляционные лаги внутри train

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

## Общие матрицы восьми горизонтов

In [ ]:
def stable_ratio(numerator, denominator):
    return numerator / np.maximum(np.abs(denominator), 1.0)


def available_statistics(window):
    valid = np.isfinite(window)
    count = valid.sum(axis=0)
    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count,
                     out=np.full(window.shape[1:], np.nan), where=count > 0)
    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count,
                         out=np.full_like(mean, np.nan), where=count > 0)
    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)
    return mean, np.sqrt(variance), spread, count


def add_experimental_features(frame, h, origins):
    additions = {}
    additions['x_time_cutoff_minutes'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))
    additions['x_profile_profile28'] = frame.profile28.to_numpy()
    additions['x_level_level24'] = level24[origins].ravel()
    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])
    current = target[origins]
    additions['x_anomaly_target_difference'] = (current - typical).ravel()
    additions['x_anomaly_target_ratio'] = stable_ratio(current, typical).ravel()
    for name in (TARGET, *STATUSES[:3]):
        short, long = means[name, 4][origins], means[name, 12][origins]
        additions[f'x_trend_{name}_difference'] = (short - long).ravel()
        additions[f'x_trend_{name}_ratio'] = stable_ratio(short, long).ravel()
        for steps in (4, 12):
            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])
            mean, std, spread, count = available_statistics(window)
            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]
            additions[f'x_volatility_{name}_std_{steps}'] = std.ravel()
            additions[f'x_volatility_{name}_range_{steps}'] = spread.ravel()
            additions[f'x_volatility_{name}_change_{steps}'] = change.ravel()
    for label, days in [('daily7', range(1,8)), ('weekly4', (7,14,21,28))]:
        window = np.stack([past(target, origins + h - day * 48) for day in days])
        mean, std, spread, count = available_statistics(window)
        additions[f'x_dispersion_{label}_std'] = std.ravel()
        additions[f'x_dispersion_{label}_relative_std'] = stable_ratio(std, mean).ravel()
        additions[f'x_dispersion_{label}_count'] = count.ravel()
    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)


def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    frame = add_experimental_features(frame, h, origins)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    if path.exists():
        print(f'h={h}: общие признаки загружены из кэша')
        return
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    print(f'h={h}: история {len(origins)*len(routes):,}, модельных признаков {len(frame.columns) - 5}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(DOCS_DIR / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origins_minutes': TRAIN_MINUTES, 'evaluation_origin': '10:30', 'correlation_storage': 'NaN placeholders; fit on stage train before model use', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'selection_end_exclusive': str(selection_end), 'all_folds': [{key: [str(period.min()), str(period.max())] for key, period in fold.items()} for fold in folds], 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))

## Метрика соревнования

In [ ]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

## Кэш, модель и Optuna

In [ ]:
import time
import joblib
import optuna
import cuml
import sklearn
import zipfile
from sklearn.impute import SimpleImputer


def atomic_json(path, value):
    temporary = path.with_suffix('.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temporary.replace(path)


def model_target(frame):
    baseline = frame.profile28.to_numpy() if RESIDUAL else 0
    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)


def transformed_frame(frame, columns):
    result = frame[columns].drop(columns='route_id').astype(np.float32)
    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1
    bits = {f'route_bit_{i}': ((codes >> i) & 1).astype(np.float32)
            for i in range(max(1, int(len(routes)).bit_length()))}
    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()


def cached_lags(boundary):
    path = OUT / f'lags_{pd.Timestamp(boundary).strftime("%Y%m%dT%H%M%S")}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    table = fit_lags(boundary)
    temporary = path.with_suffix('.partial')
    table.to_parquet(temporary, index=False)
    temporary.replace(path)
    return table


def prepare_stage(h, stage, train_dates=None, validation_dates=None, boundary=None):
    folder = PREPARED / f'h{h}_{stage}'
    marker = folder / 'ready.json'
    if marker.exists():
        return folder
    folder.mkdir(parents=True, exist_ok=True)
    filters = [('timestamp', '<', selection_end)] if stage.startswith('fold') else None
    history = pd.read_parquet(OUT / f'history_h{h}.parquet', filters=filters)
    history = history.loc[np.isfinite(history.profile28)].copy()
    mask = history.timestamp < boundary
    if train_dates is not None:
        mask &= history.cutoff.dt.normalize().isin(train_dates.normalize())
    train = history.loc[mask].copy()
    valid = history.loc[history.cutoff.isin(validation_dates)].copy()
    del history
    if train.empty or valid.empty or train.timestamp.max() >= valid.cutoff.min().normalize():
        raise ValueError('Нарушено временное разделение')
    if stage.startswith('fold') and valid.timestamp.max() >= selection_end:
        raise ValueError('Tuning выходит за границу подбора')
    table = cached_lags(boundary)
    train = add_correlation_features(train, h, table)
    valid = add_correlation_features(valid, h, table)
    columns = model_columns(train)
    prep = SimpleImputer(strategy='median', keep_empty_features=True)
    for name, frame in [('train', train), ('valid', valid)]:
        encoded = transformed_frame(frame, columns)
        matrix = prep.fit_transform(encoded) if name == 'train' else prep.transform(encoded)
        np.save(folder / f'{name}.npy', np.asarray(matrix, dtype=np.float32))
        frame[['cutoff', 'timestamp', 'route_id', 'y', 'profile28']].to_parquet(folder / f'{name}.parquet', index=False)
        del matrix, encoded
    joblib.dump(prep, folder / 'preprocess.joblib')
    atomic_json(marker, dict(columns=columns, boundary=str(boundary), rows_train=len(train), rows_valid=len(valid)))
    return folder


def load_stage(folder):
    return {name: (pd.read_parquet(folder / f'{name}.parquet'), np.load(folder / f'{name}.npy', mmap_mode='r')) for name in ('train', 'valid')}


def fit_forest(frame, matrix, params):
    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type='numpy', **params)
    model.fit(matrix, model_target(frame))
    cp.cuda.get_current_stream().synchronize()
    return model


def predict_matrix(model, frame, matrix):
    result = np.empty(len(frame), dtype=np.float64)
    for start in range(0, len(frame), PREDICTION_BATCH):
        stop = min(start + PREDICTION_BATCH, len(frame))
        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)
        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0
        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)
    if not np.isfinite(result).all():
        raise ValueError('Некорректный прогноз')
    return result


def prediction_part(frame, prediction, h, period):
    part = frame.copy()
    part['prediction'] = prediction
    part[ALGORITHM] = prediction
    part['horizon'] = h
    part['period'] = period
    return part


def report(rows, phase, h, trial=None, fold=None, seconds=None):
    records = [dict(phase=phase, horizon=h, trial=trial, fold=fold, seconds=seconds, **row) for row in rows]
    path = DOCS_DIR / f'{phase}_h{h}_trial{trial}_fold{fold}.json'
    atomic_json(path, records)
    display(pd.DataFrame(records))


def suggest_params(trial):
    return dict(n_estimators=trial.suggest_categorical('n_estimators', RF_TREES),
                max_depth=trial.suggest_categorical('max_depth', RF_DEPTHS),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 8, 256, log=True),
                max_features=trial.suggest_float('max_features', 0.3, 0.9),
                max_samples=trial.suggest_float('max_samples', 0.5, 1))


def select_horizon(h):
    sampler_path = POSTPROCESS_DIR / f'sampler_h{h}.joblib'
    sampler = joblib.load(sampler_path) if sampler_path.exists() else optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler,
                               storage=f'sqlite:///{(POSTPROCESS_DIR / "search.sqlite3").resolve().as_posix()}',
                               study_name=f'h{h}', load_if_exists=True)
    if not study.trials:
        study.enqueue_trial(base_params_for_horizon(h))
    for trial in study.get_trials(states=(optuna.trial.TrialState.RUNNING,)):
        study.tell(trial.number, state=optuna.trial.TrialState.FAIL)
        if trial.params:
            study.enqueue_trial(trial.params, skip_if_exists=False)
    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
    remaining = max(0, N_TRIALS - len(completed))
    folders = []
    if remaining:
        for i, fold in enumerate(folds):
            folders.append(prepare_stage(h, f'fold{i+1}', fold['train'], fold['tuning'], fold['val'][0].normalize()))
    def objective(trial):
        started = time.perf_counter()
        params = suggest_params(trial)
        scores, parts = [], []
        for i, folder in enumerate(folders):
            fold_started = time.perf_counter()
            data = load_stage(folder)
            train, x_train = data['train']
            tuning, x_tuning = data['valid']
            model = fit_forest(train, x_train, params)
            prediction = predict_matrix(model, tuning, x_tuning)
            rows = [score_row(train, predict_matrix(model, train, x_train), 'train'), score_row(tuning, prediction, 'tuning')]
            scores.append(rows[-1]['метрика конкурса'])
            part = prediction_part(tuning, prediction, h, 'tuning')
            part['fold'] = i + 1
            part['fit_labels_before'] = folds[i]['val'][0].normalize()
            part['selection_labels_through'] = folds[-1]['tuning'][-1] + h * STEP
            part['model_info_through'] = part['selection_labels_through']
            part['is_unbiased_oof'] = False
            parts.append(part)
            report(rows, 'search', h, trial.number, i + 1, time.perf_counter() - fold_started)
            del model, data, train, tuning, x_train, x_tuning
            gc.collect()
        value = float(np.mean(scores))
        if not np.isfinite(value):
            raise ValueError('Неконечная метрика подбора')
        path = POSTPROCESS_DIR / f'tuning_h{h}_trial{trial.number}.parquet'
        temporary = path.with_suffix('.partial')
        pd.concat(parts, ignore_index=True).to_parquet(temporary, index=False)
        temporary.replace(path)
        trial.set_user_attr('fold_scores', scores)
        trial.set_user_attr('elapsed_seconds', time.perf_counter() - started)
        return value
    def save_sampler(study, trial):
        temporary = sampler_path.with_suffix('.partial')
        joblib.dump(study.sampler, temporary)
        temporary.replace(sampler_path)
    if remaining:
        study.optimize(objective, n_trials=remaining, n_jobs=1, gc_after_trial=True, callbacks=[save_sampler])
    study.trials_dataframe().to_csv(DOCS_DIR / f'trials_h{h}.csv', index=False)
    best = study.best_trial
    best_part = pd.read_parquet(POSTPROCESS_DIR / f'tuning_h{h}_trial{best.number}.parquet')
    best_part.to_parquet(POSTPROCESS_DIR / f'tuning_predictions_h{h}.parquet', index=False)
    chosen = dict(algorithm=ALGORITHM, residual=RESIDUAL, horizon=h, params=best.params, trial=best.number,
                  tuning_score=best.value, fold_scores=best.user_attrs['fold_scores'], trees=best.params['n_estimators'],
                  selection_end_exclusive=str(selection_end), selection_labels_through=str(folds[-1]['tuning'][-1] + h * STEP))
    atomic_json(PARAMS_DIR / f'best_h{h}.json', chosen)
    durations = [t.user_attrs['elapsed_seconds'] for t in study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))]
    print(f'h={h}: завершено {len(durations)} trials, среднее {np.mean(durations)/60:.1f} мин/trial')
    return chosen




In [ ]:
saved_comparisons = json.loads(r'''{"direct/baseline/h1_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.256588,"Relative Bias":-0.000261,"|Relative Bias|":0.000261,"метрика конкурса":0.256849,"среднее по дням":0.264723,"variant":"direct","experiment":"baseline","horizon":1,"fold":1,"features":388,"volume":29748780000.0,"absolute_error":7633171000.0,"signed_error":-7766953.0,"seconds":26.22239},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262693,"Relative Bias":0.005146,"|Relative Bias|":0.005146,"метрика конкурса":0.267838,"среднее по дням":0.288929,"variant":"direct","experiment":"baseline","horizon":1,"fold":1,"features":388,"volume":1932004000.0,"absolute_error":507523200.0,"signed_error":9941647.0,"seconds":26.245869}],"direct/thin_offsets/h1_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.258224,"Relative Bias":-0.000313,"|Relative Bias|":0.000313,"метрика конкурса":0.258536,"среднее по дням":0.266892,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":1,"features":232,"volume":29748780000.0,"absolute_error":7681844000.0,"signed_error":-9298659.0,"seconds":9.917906},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262388,"Relative Bias":0.004355,"|Relative Bias|":0.004355,"метрика конкурса":0.266743,"среднее по дням":0.288279,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":1,"features":232,"volume":1932004000.0,"absolute_error":506934700.0,"signed_error":8413071.0,"seconds":9.933219}],"direct/compact_with_additions/h1_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.257013,"Relative Bias":-0.000266,"|Relative Bias|":0.000266,"метрика конкурса":0.257279,"среднее по дням":0.264974,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":1,"features":245,"volume":29748780000.0,"absolute_error":7645828000.0,"signed_error":-7922351.0,"seconds":10.176453},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.262095,"Relative Bias":0.006041,"|Relative Bias|":0.006041,"метрика конкурса":0.268136,"среднее по дням":0.289183,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":1,"features":245,"volume":1932004000.0,"absolute_error":506367700.0,"signed_error":11672040.0,"seconds":10.187476}],"direct/baseline/h1_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.257883,"Relative Bias":-0.000292,"|Relative Bias|":0.000292,"метрика конкурса":0.258175,"среднее по дням":0.266516,"variant":"direct","experiment":"baseline","horizon":1,"fold":2,"features":388,"volume":38649630000.0,"absolute_error":9967092000.0,"signed_error":-11277700.0,"seconds":33.672311},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.283222,"Relative Bias":0.066929,"|Relative Bias|":0.066929,"метрика конкурса":0.350151,"среднее по дням":0.365136,"variant":"direct","experiment":"baseline","horizon":1,"fold":2,"features":388,"volume":1903508000.0,"absolute_error":539115000.0,"signed_error":127399400.0,"seconds":33.686869}],"direct/thin_offsets/h1_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.259491,"Relative Bias":-0.000252,"|Relative Bias|":0.000252,"метрика конкурса":0.259743,"среднее по дням":0.269175,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":2,"features":232,"volume":38649630000.0,"absolute_error":10029240000.0,"signed_error":-9750086.0,"seconds":14.976312},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.282792,"Relative Bias":0.064997,"|Relative Bias|":0.064997,"метрика конкурса":0.34779,"среднее по дням":0.362752,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":2,"features":232,"volume":1903508000.0,"absolute_error":538297500.0,"signed_error":123723200.0,"seconds":14.986894}],"direct/compact_with_additions/h1_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.25828,"Relative Bias":-0.000236,"|Relative Bias|":0.000236,"метрика конкурса":0.258516,"среднее по дням":0.266571,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":2,"features":245,"volume":38649630000.0,"absolute_error":9982411000.0,"signed_error":-9129765.0,"seconds":15.221267},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.282914,"Relative Bias":0.067732,"|Relative Bias|":0.067732,"метрика конкурса":0.350646,"среднее по дням":0.36579,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":2,"features":245,"volume":1903508000.0,"absolute_error":538530100.0,"signed_error":128927800.0,"seconds":15.232375}],"direct/baseline/h1_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.258536,"Relative Bias":-0.000325,"|Relative Bias|":0.000325,"метрика конкурса":0.258861,"среднее по дням":0.268499,"variant":"direct","experiment":"baseline","horizon":1,"fold":3,"features":388,"volume":48027310000.0,"absolute_error":12416790000.0,"signed_error":-15625660.0,"seconds":41.941955},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.264405,"Relative Bias":-0.01111,"|Relative Bias|":0.01111,"метрика конкурса":0.275516,"среднее по дням":0.281283,"variant":"direct","experiment":"baseline","horizon":1,"fold":3,"features":388,"volume":2053871000.0,"absolute_error":543054000.0,"signed_error":-22819470.0,"seconds":41.956384}],"direct/thin_offsets/h1_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.259918,"Relative Bias":-0.000364,"|Relative Bias|":0.000364,"метрика конкурса":0.260282,"среднее по дням":0.270764,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":3,"features":232,"volume":48027310000.0,"absolute_error":12483160000.0,"signed_error":-17488320.0,"seconds":18.321837},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.264843,"Relative Bias":-0.009796,"|Relative Bias|":0.009796,"метрика конкурса":0.27464,"среднее по дням":0.281483,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":3,"features":232,"volume":2053871000.0,"absolute_error":543954000.0,"signed_error":-20120260.0,"seconds":18.333916}],"direct/compact_with_additions/h1_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.258686,"Relative Bias":-0.000309,"|Relative Bias|":0.000309,"метрика конкурса":0.258994,"среднее по дням":0.268537,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":3,"features":245,"volume":48027310000.0,"absolute_error":12423970000.0,"signed_error":-14826220.0,"seconds":17.456716},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.263485,"Relative Bias":-0.007273,"|Relative Bias|":0.007273,"метрика конкурса":0.270758,"среднее по дням":0.278013,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":3,"features":245,"volume":2053871000.0,"absolute_error":541164400.0,"signed_error":-14938430.0,"seconds":17.468464}],"direct/baseline/h1_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.259515,"Relative Bias":-0.000144,"|Relative Bias|":0.000144,"метрика конкурса":0.259659,"среднее по дням":0.270522,"variant":"direct","experiment":"baseline","horizon":1,"fold":4,"features":388,"volume":57620340000.0,"absolute_error":14953370000.0,"signed_error":-8274031.0,"seconds":48.856591},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.259797,"Relative Bias":-0.036723,"|Relative Bias|":0.036723,"метрика конкурса":0.29652,"среднее по дням":0.295791,"variant":"direct","experiment":"baseline","horizon":1,"fold":4,"features":388,"volume":2063192000.0,"absolute_error":536011200.0,"signed_error":-75767570.0,"seconds":48.871821}],"direct/thin_offsets/h1_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.260822,"Relative Bias":-0.00017,"|Relative Bias|":0.00017,"метрика конкурса":0.260993,"среднее по дням":0.272434,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":4,"features":232,"volume":57620340000.0,"absolute_error":15028670000.0,"signed_error":-9806383.0,"seconds":21.881792},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.260363,"Relative Bias":-0.029393,"|Relative Bias|":0.029393,"метрика конкурса":0.289756,"среднее по дням":0.291913,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":4,"features":232,"volume":2063192000.0,"absolute_error":537179000.0,"signed_error":-60642810.0,"seconds":21.892805}],"direct/compact_with_additions/h1_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.259667,"Relative Bias":-0.000136,"|Relative Bias|":0.000136,"метрика конкурса":0.259803,"среднее по дням":0.271286,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":4,"features":245,"volume":57620340000.0,"absolute_error":14962080000.0,"signed_error":-7846767.0,"seconds":20.324936},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.258985,"Relative Bias":-0.032566,"|Relative Bias|":0.032566,"метрика конкурса":0.291551,"среднее по дням":0.29216,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":4,"features":245,"volume":2063192000.0,"absolute_error":534335700.0,"signed_error":-67189490.0,"seconds":20.336294}],"direct/baseline/h1_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.260327,"Relative Bias":-0.000313,"|Relative Bias|":0.000313,"метрика конкурса":0.260639,"среднее по дням":0.272319,"variant":"direct","experiment":"baseline","horizon":1,"fold":5,"features":388,"volume":67659790000.0,"absolute_error":17613670000.0,"signed_error":-21147150.0,"seconds":58.254558},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.268209,"Relative Bias":-0.006437,"|Relative Bias|":0.006437,"метрика конкурса":0.274646,"среднее по дням":0.291013,"variant":"direct","experiment":"baseline","horizon":1,"fold":5,"features":388,"volume":2075688000.0,"absolute_error":556718600.0,"signed_error":-13361290.0,"seconds":58.268711}],"direct/thin_offsets/h1_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.26154,"Relative Bias":-0.000271,"|Relative Bias|":0.000271,"метрика конкурса":0.261811,"среднее по дням":0.274039,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":5,"features":232,"volume":67659790000.0,"absolute_error":17695760000.0,"signed_error":-18347970.0,"seconds":23.771553},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.26948,"Relative Bias":-0.002641,"|Relative Bias|":0.002641,"метрика конкурса":0.272122,"среднее по дням":0.293276,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":5,"features":232,"volume":2075688000.0,"absolute_error":559356900.0,"signed_error":-5482548.0,"seconds":23.782195}],"direct/compact_with_additions/h1_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.26029,"Relative Bias":-0.000195,"|Relative Bias|":0.000195,"метрика конкурса":0.260485,"среднее по дням":0.272435,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":5,"features":245,"volume":67659790000.0,"absolute_error":17611180000.0,"signed_error":-13193000.0,"seconds":23.176182},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.267897,"Relative Bias":-0.003575,"|Relative Bias|":0.003575,"метрика конкурса":0.271471,"среднее по дням":0.291541,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":5,"features":245,"volume":2075688000.0,"absolute_error":556070000.0,"signed_error":-7419993.0,"seconds":23.187292}],"direct/baseline/h1_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.260789,"Relative Bias":-0.000214,"|Relative Bias|":0.000214,"метрика конкурса":0.261002,"среднее по дням":0.274907,"variant":"direct","experiment":"baseline","horizon":1,"fold":6,"features":388,"volume":77733690000.0,"absolute_error":20272060000.0,"signed_error":-16621450.0,"seconds":66.921942},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.27028,"Relative Bias":-0.00915,"|Relative Bias|":0.00915,"метрика конкурса":0.27943,"среднее по дням":0.290103,"variant":"direct","experiment":"baseline","horizon":1,"fold":6,"features":388,"volume":2084706000.0,"absolute_error":563454000.0,"signed_error":-19075960.0,"seconds":66.93644}],"direct/thin_offsets/h1_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.261845,"Relative Bias":-0.000248,"|Relative Bias|":0.000248,"метрика конкурса":0.262094,"среднее по дням":0.275854,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":6,"features":232,"volume":77733690000.0,"absolute_error":20354210000.0,"signed_error":-19295440.0,"seconds":26.577312},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.271777,"Relative Bias":-0.003451,"|Relative Bias|":0.003451,"метрика конкурса":0.275228,"среднее по дням":0.293399,"variant":"direct","experiment":"thin_offsets","horizon":1,"fold":6,"features":232,"volume":2084706000.0,"absolute_error":566574000.0,"signed_error":-7195277.0,"seconds":26.588443}],"direct/compact_with_additions/h1_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.260668,"Relative Bias":-0.000156,"|Relative Bias|":0.000156,"метрика конкурса":0.260823,"среднее по дням":0.274446,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":6,"features":245,"volume":77733690000.0,"absolute_error":20262680000.0,"signed_error":-12087990.0,"seconds":27.515281},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.269882,"Relative Bias":-0.008092,"|Relative Bias|":0.008092,"метрика конкурса":0.277974,"среднее по дням":0.290694,"variant":"direct","experiment":"compact_with_additions","horizon":1,"fold":6,"features":245,"volume":2084706000.0,"absolute_error":562624900.0,"signed_error":-16868680.0,"seconds":27.525926}],"direct/baseline/h2_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.322831,"Relative Bias":-8.7e-05,"|Relative Bias|":8.7e-05,"метрика конкурса":0.322918,"среднее по дням":0.337179,"variant":"direct","experiment":"baseline","horizon":2,"fold":1,"features":390,"volume":30245590000.0,"absolute_error":9764216000.0,"signed_error":-2631093.0,"seconds":25.698697},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325589,"Relative Bias":0.006297,"|Relative Bias|":0.006297,"метрика конкурса":0.331886,"среднее по дням":0.344449,"variant":"direct","experiment":"baseline","horizon":2,"fold":1,"features":390,"volume":1915804000.0,"absolute_error":623765300.0,"signed_error":12064080.0,"seconds":25.713429}],"direct/thin_offsets/h2_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.323415,"Relative Bias":-0.000126,"|Relative Bias|":0.000126,"метрика конкурса":0.32354,"среднее по дням":0.338257,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":1,"features":234,"volume":30245590000.0,"absolute_error":9781864000.0,"signed_error":-3801653.0,"seconds":11.521686},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325618,"Relative Bias":0.005353,"|Relative Bias|":0.005353,"метрика конкурса":0.330971,"среднее по дням":0.345024,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":1,"features":234,"volume":1915804000.0,"absolute_error":623820800.0,"signed_error":10255820.0,"seconds":11.532506}],"direct/compact_with_additions/h2_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.321371,"Relative Bias":-0.000128,"|Relative Bias|":0.000128,"метрика конкурса":0.321499,"среднее по дням":0.33589,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":1,"features":247,"volume":30245590000.0,"absolute_error":9720049000.0,"signed_error":-3882704.0,"seconds":10.958094},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324349,"Relative Bias":0.009249,"|Relative Bias|":0.009249,"метрика конкурса":0.333599,"среднее по дням":0.346471,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":1,"features":247,"volume":1915804000.0,"absolute_error":621389700.0,"signed_error":17719700.0,"seconds":10.969433}],"direct/baseline/h2_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.322139,"Relative Bias":-0.000287,"|Relative Bias|":0.000287,"метрика конкурса":0.322426,"среднее по дням":0.33868,"variant":"direct","experiment":"baseline","horizon":2,"fold":2,"features":390,"volume":39292040000.0,"absolute_error":12657480000.0,"signed_error":-11290050.0,"seconds":31.786005},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330877,"Relative Bias":0.056523,"|Relative Bias|":0.056523,"метрика конкурса":0.3874,"среднее по дням":0.395405,"variant":"direct","experiment":"baseline","horizon":2,"fold":2,"features":390,"volume":1933559000.0,"absolute_error":639770300.0,"signed_error":109291100.0,"seconds":31.800458}],"direct/thin_offsets/h2_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.322582,"Relative Bias":-0.000275,"|Relative Bias|":0.000275,"метрика конкурса":0.322857,"среднее по дням":0.339777,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":2,"features":234,"volume":39292040000.0,"absolute_error":12674890000.0,"signed_error":-10819120.0,"seconds":14.40306},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330706,"Relative Bias":0.056268,"|Relative Bias|":0.056268,"метрика конкурса":0.386974,"среднее по дням":0.394984,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":2,"features":234,"volume":1933559000.0,"absolute_error":639439800.0,"signed_error":108798100.0,"seconds":14.414307}],"direct/compact_with_additions/h2_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.320532,"Relative Bias":-0.00027,"|Relative Bias|":0.00027,"метрика конкурса":0.320803,"среднее по дням":0.336607,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":2,"features":247,"volume":39292040000.0,"absolute_error":12594370000.0,"signed_error":-10623560.0,"seconds":14.296427},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330885,"Relative Bias":0.066045,"|Relative Bias|":0.066045,"метрика конкурса":0.39693,"среднее по дням":0.403121,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":2,"features":247,"volume":1933559000.0,"absolute_error":639785800.0,"signed_error":127701700.0,"seconds":14.307703}],"direct/baseline/h2_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.320997,"Relative Bias":-0.000197,"|Relative Bias|":0.000197,"метрика конкурса":0.321195,"среднее по дням":0.338819,"variant":"direct","experiment":"baseline","horizon":2,"fold":3,"features":390,"volume":48881010000.0,"absolute_error":15690670000.0,"signed_error":-9640998.0,"seconds":38.937374},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.314283,"Relative Bias":-0.039334,"|Relative Bias|":0.039334,"метрика конкурса":0.353616,"среднее по дням":0.364575,"variant":"direct","experiment":"baseline","horizon":2,"fold":3,"features":390,"volume":2101718000.0,"absolute_error":660533400.0,"signed_error":-82668000.0,"seconds":38.952366}],"direct/thin_offsets/h2_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.321237,"Relative Bias":-0.000186,"|Relative Bias|":0.000186,"метрика конкурса":0.321424,"среднее по дням":0.33959,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":3,"features":234,"volume":48881010000.0,"absolute_error":15702400000.0,"signed_error":-9109512.0,"seconds":17.409773},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.314248,"Relative Bias":-0.038177,"|Relative Bias|":0.038177,"метрика конкурса":0.352425,"среднее по дням":0.363744,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":3,"features":234,"volume":2101718000.0,"absolute_error":660460800.0,"signed_error":-80237770.0,"seconds":17.421314}],"direct/compact_with_additions/h2_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.319291,"Relative Bias":-0.00022,"|Relative Bias|":0.00022,"метрика конкурса":0.319511,"среднее по дням":0.337395,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":3,"features":247,"volume":48881010000.0,"absolute_error":15607250000.0,"signed_error":-10765940.0,"seconds":17.077481},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.312016,"Relative Bias":-0.030491,"|Relative Bias|":0.030491,"метрика конкурса":0.342507,"среднее по дням":0.355148,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":3,"features":247,"volume":2101718000.0,"absolute_error":655769200.0,"signed_error":-64084220.0,"seconds":17.088973}],"direct/baseline/h2_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.32077,"Relative Bias":-0.000112,"|Relative Bias|":0.000112,"метрика конкурса":0.320882,"среднее по дням":0.339687,"variant":"direct","experiment":"baseline","horizon":2,"fold":4,"features":390,"volume":58796850000.0,"absolute_error":18860250000.0,"signed_error":-6578303.0,"seconds":46.300561},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325713,"Relative Bias":-0.03971,"|Relative Bias|":0.03971,"метрика конкурса":0.365423,"среднее по дням":0.374232,"variant":"direct","experiment":"baseline","horizon":2,"fold":4,"features":390,"volume":2057112000.0,"absolute_error":670028800.0,"signed_error":-81688010.0,"seconds":46.315267}],"direct/thin_offsets/h2_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.320991,"Relative Bias":-8.2e-05,"|Relative Bias|":8.2e-05,"метрика конкурса":0.321072,"среднее по дням":0.34051,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":4,"features":234,"volume":58796850000.0,"absolute_error":18873240000.0,"signed_error":-4801584.0,"seconds":20.732989},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325618,"Relative Bias":-0.036691,"|Relative Bias|":0.036691,"метрика конкурса":0.362308,"среднее по дням":0.373428,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":4,"features":234,"volume":2057112000.0,"absolute_error":669831800.0,"signed_error":-75476730.0,"seconds":20.744054}],"direct/compact_with_additions/h2_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.319212,"Relative Bias":-0.00011,"|Relative Bias|":0.00011,"метрика конкурса":0.319322,"среднее по дням":0.338986,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":4,"features":247,"volume":58796850000.0,"absolute_error":18768650000.0,"signed_error":-6490754.0,"seconds":20.025272},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323687,"Relative Bias":-0.037542,"|Relative Bias|":0.037542,"метрика конкурса":0.361229,"среднее по дням":0.371178,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":4,"features":247,"volume":2057112000.0,"absolute_error":665860800.0,"signed_error":-77227730.0,"seconds":20.036736}],"direct/baseline/h2_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.320721,"Relative Bias":-0.000116,"|Relative Bias|":0.000116,"метрика конкурса":0.320837,"среднее по дням":0.341301,"variant":"direct","experiment":"baseline","horizon":2,"fold":5,"features":390,"volume":69126560000.0,"absolute_error":22170370000.0,"signed_error":-8020000.0,"seconds":53.884605},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.317186,"Relative Bias":-0.015417,"|Relative Bias|":0.015417,"метрика конкурса":0.332603,"среднее по дням":0.342247,"variant":"direct","experiment":"baseline","horizon":2,"fold":5,"features":390,"volume":2118564000.0,"absolute_error":671978800.0,"signed_error":-32661950.0,"seconds":53.899636}],"direct/thin_offsets/h2_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.320847,"Relative Bias":-0.000114,"|Relative Bias|":0.000114,"метрика конкурса":0.320962,"среднее по дням":0.341898,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":5,"features":234,"volume":69126560000.0,"absolute_error":22179070000.0,"signed_error":-7901577.0,"seconds":23.530924},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.317211,"Relative Bias":-0.013345,"|Relative Bias|":0.013345,"метрика конкурса":0.330556,"среднее по дням":0.341094,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":5,"features":234,"volume":2118564000.0,"absolute_error":672031800.0,"signed_error":-28272670.0,"seconds":23.542064}],"direct/compact_with_additions/h2_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.319037,"Relative Bias":-0.000124,"|Relative Bias|":0.000124,"метрика конкурса":0.31916,"среднее по дням":0.340092,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":5,"features":247,"volume":69126560000.0,"absolute_error":22053900000.0,"signed_error":-8554299.0,"seconds":23.178056},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.315049,"Relative Bias":-0.013012,"|Relative Bias|":0.013012,"метрика конкурса":0.328061,"среднее по дням":0.339498,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":5,"features":247,"volume":2118564000.0,"absolute_error":667452100.0,"signed_error":-27566160.0,"seconds":23.18974}],"direct/baseline/h2_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.321427,"Relative Bias":-0.000107,"|Relative Bias|":0.000107,"метрика конкурса":0.321534,"среднее по дням":0.344727,"variant":"direct","experiment":"baseline","horizon":2,"fold":6,"features":390,"volume":79480170000.0,"absolute_error":25547080000.0,"signed_error":-8492330.0,"seconds":62.926732},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322175,"Relative Bias":-0.00411,"|Relative Bias|":0.00411,"метрика конкурса":0.326285,"среднее по дням":0.343481,"variant":"direct","experiment":"baseline","horizon":2,"fold":6,"features":390,"volume":2122104000.0,"absolute_error":683689800.0,"signed_error":-8721293.0,"seconds":62.941918}],"direct/thin_offsets/h2_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.321518,"Relative Bias":-9.9e-05,"|Relative Bias|":9.9e-05,"метрика конкурса":0.321617,"среднее по дням":0.344884,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":6,"features":234,"volume":79480170000.0,"absolute_error":25554310000.0,"signed_error":-7876271.0,"seconds":26.348142},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322423,"Relative Bias":-0.001597,"|Relative Bias|":0.001597,"метрика конкурса":0.32402,"среднее по дням":0.343666,"variant":"direct","experiment":"thin_offsets","horizon":2,"fold":6,"features":234,"volume":2122104000.0,"absolute_error":684216000.0,"signed_error":-3388501.0,"seconds":26.358649}],"direct/compact_with_additions/h2_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.319735,"Relative Bias":-8.8e-05,"|Relative Bias|":8.8e-05,"метрика конкурса":0.319823,"среднее по дням":0.343264,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":6,"features":247,"volume":79480170000.0,"absolute_error":25412590000.0,"signed_error":-6973333.0,"seconds":26.802039},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.320224,"Relative Bias":-0.005358,"|Relative Bias|":0.005358,"метрика конкурса":0.325582,"среднее по дням":0.343564,"variant":"direct","experiment":"compact_with_additions","horizon":2,"fold":6,"features":247,"volume":2122104000.0,"absolute_error":679549700.0,"signed_error":-11370250.0,"seconds":26.813156}],"direct/baseline/h3_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334374,"Relative Bias":-0.000204,"|Relative Bias|":0.000204,"метрика конкурса":0.334578,"среднее по дням":0.34946,"variant":"direct","experiment":"baseline","horizon":3,"fold":1,"features":392,"volume":30060130000.0,"absolute_error":10051320000.0,"signed_error":-6143330.0,"seconds":25.961082},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.321662,"Relative Bias":0.00286,"|Relative Bias|":0.00286,"метрика конкурса":0.324522,"среднее по дням":0.343846,"variant":"direct","experiment":"baseline","horizon":3,"fold":1,"features":392,"volume":1931182000.0,"absolute_error":621188200.0,"signed_error":5522907.0,"seconds":25.976591}],"direct/thin_offsets/h3_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334701,"Relative Bias":-0.000257,"|Relative Bias|":0.000257,"метрика конкурса":0.334958,"среднее по дням":0.349629,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":1,"features":236,"volume":30060130000.0,"absolute_error":10061150000.0,"signed_error":-7718776.0,"seconds":11.569985},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322041,"Relative Bias":0.004142,"|Relative Bias|":0.004142,"метрика конкурса":0.326183,"среднее по дням":0.344883,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":1,"features":236,"volume":1931182000.0,"absolute_error":621919800.0,"signed_error":7999500.0,"seconds":11.580688}],"direct/compact_with_additions/h3_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333435,"Relative Bias":-0.00023,"|Relative Bias|":0.00023,"метрика конкурса":0.333664,"среднее по дням":0.349937,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":1,"features":249,"volume":30060130000.0,"absolute_error":10023090000.0,"signed_error":-6904869.0,"seconds":11.043004},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.321068,"Relative Bias":0.007096,"|Relative Bias|":0.007096,"метрика конкурса":0.328163,"среднее по дням":0.348329,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":1,"features":249,"volume":1931182000.0,"absolute_error":620039900.0,"signed_error":13702780.0,"seconds":11.054471}],"direct/baseline/h3_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333386,"Relative Bias":-0.000282,"|Relative Bias|":0.000282,"метрика конкурса":0.333669,"среднее по дням":0.349003,"variant":"direct","experiment":"baseline","horizon":3,"fold":2,"features":392,"volume":39081400000.0,"absolute_error":13029210000.0,"signed_error":-11038620.0,"seconds":32.005029},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.316772,"Relative Bias":-0.00223,"|Relative Bias|":0.00223,"метрика конкурса":0.319003,"среднее по дням":0.361776,"variant":"direct","experiment":"baseline","horizon":3,"fold":2,"features":392,"volume":2064414000.0,"absolute_error":653949600.0,"signed_error":-4604600.0,"seconds":32.020142}],"direct/thin_offsets/h3_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333623,"Relative Bias":-0.000245,"|Relative Bias|":0.000245,"метрика конкурса":0.333868,"среднее по дням":0.349654,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":2,"features":236,"volume":39081400000.0,"absolute_error":13038450000.0,"signed_error":-9590512.0,"seconds":14.512181},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.316888,"Relative Bias":-0.001118,"|Relative Bias|":0.001118,"метрика конкурса":0.318006,"среднее по дням":0.361741,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":2,"features":236,"volume":2064414000.0,"absolute_error":654187800.0,"signed_error":-2307710.0,"seconds":14.52361}],"direct/compact_with_additions/h3_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332387,"Relative Bias":-0.000251,"|Relative Bias|":0.000251,"метрика конкурса":0.332638,"среднее по дням":0.348604,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":2,"features":249,"volume":39081400000.0,"absolute_error":12990150000.0,"signed_error":-9802305.0,"seconds":14.201775},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.315856,"Relative Bias":0.004984,"|Relative Bias|":0.004984,"метрика конкурса":0.32084,"среднее по дням":0.362536,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":2,"features":249,"volume":2064414000.0,"absolute_error":652057100.0,"signed_error":10288800.0,"seconds":14.212978}],"direct/baseline/h3_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.33174,"Relative Bias":-0.000179,"|Relative Bias|":0.000179,"метрика конкурса":0.33192,"среднее по дням":0.347899,"variant":"direct","experiment":"baseline","horizon":3,"fold":3,"features":392,"volume":48698000000.0,"absolute_error":16155090000.0,"signed_error":-8734305.0,"seconds":38.90073},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325419,"Relative Bias":-0.034284,"|Relative Bias|":0.034284,"метрика конкурса":0.359703,"среднее по дням":0.371215,"variant":"direct","experiment":"baseline","horizon":3,"fold":3,"features":392,"volume":2118422000.0,"absolute_error":689374500.0,"signed_error":-72627190.0,"seconds":38.915511}],"direct/thin_offsets/h3_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.332007,"Relative Bias":-0.000173,"|Relative Bias|":0.000173,"метрика конкурса":0.33218,"среднее по дням":0.348451,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":3,"features":236,"volume":48698000000.0,"absolute_error":16168090000.0,"signed_error":-8415185.0,"seconds":17.553912},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325634,"Relative Bias":-0.031363,"|Relative Bias|":0.031363,"метрика конкурса":0.356997,"среднее по дням":0.36939,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":3,"features":236,"volume":2118422000.0,"absolute_error":689829300.0,"signed_error":-66440050.0,"seconds":17.565926}],"direct/compact_with_additions/h3_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330804,"Relative Bias":-0.00018,"|Relative Bias|":0.00018,"метрика конкурса":0.330984,"среднее по дням":0.348013,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":3,"features":249,"volume":48698000000.0,"absolute_error":16109490000.0,"signed_error":-8779546.0,"seconds":16.999582},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324496,"Relative Bias":-0.026203,"|Relative Bias|":0.026203,"метрика конкурса":0.350699,"среднее по дням":0.362524,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":3,"features":249,"volume":2118422000.0,"absolute_error":687419000.0,"signed_error":-55509570.0,"seconds":17.011177}],"direct/baseline/h3_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.331085,"Relative Bias":-7.9e-05,"|Relative Bias|":7.9e-05,"метрика конкурса":0.331165,"среднее по дням":0.347925,"variant":"direct","experiment":"baseline","horizon":3,"fold":4,"features":392,"volume":58727020000.0,"absolute_error":19443670000.0,"signed_error":-4647942.0,"seconds":46.243536},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328504,"Relative Bias":-0.056152,"|Relative Bias|":0.056152,"метрика конкурса":0.384656,"среднее по дням":0.384134,"variant":"direct","experiment":"baseline","horizon":3,"fold":4,"features":392,"volume":2120046000.0,"absolute_error":696444500.0,"signed_error":-119044300.0,"seconds":46.258174}],"direct/thin_offsets/h3_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.331315,"Relative Bias":-8.1e-05,"|Relative Bias|":8.1e-05,"метрика конкурса":0.331396,"среднее по дням":0.348371,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":4,"features":236,"volume":58727020000.0,"absolute_error":19457160000.0,"signed_error":-4744659.0,"seconds":20.941715},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328281,"Relative Bias":-0.051016,"|Relative Bias|":0.051016,"метрика конкурса":0.379297,"среднее по дням":0.378762,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":4,"features":236,"volume":2120046000.0,"absolute_error":695970400.0,"signed_error":-108156300.0,"seconds":20.952528}],"direct/compact_with_additions/h3_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.330263,"Relative Bias":-4.9e-05,"|Relative Bias|":4.9e-05,"метрика конкурса":0.330311,"среднее по дням":0.348337,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":4,"features":249,"volume":58727020000.0,"absolute_error":19395340000.0,"signed_error":-2863123.0,"seconds":19.975772},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326993,"Relative Bias":-0.050668,"|Relative Bias|":0.050668,"метрика конкурса":0.377661,"среднее по дням":0.377153,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":4,"features":249,"volume":2120046000.0,"absolute_error":693240400.0,"signed_error":-107418200.0,"seconds":19.987443}],"direct/baseline/h3_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.330459,"Relative Bias":-9.5e-05,"|Relative Bias|":9.5e-05,"метрика конкурса":0.330554,"среднее по дням":0.349168,"variant":"direct","experiment":"baseline","horizon":3,"fold":5,"features":392,"volume":69199000000.0,"absolute_error":22867440000.0,"signed_error":-6545585.0,"seconds":54.890009},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325333,"Relative Bias":-0.018886,"|Relative Bias|":0.018886,"метрика конкурса":0.344219,"среднее по дням":0.363458,"variant":"direct","experiment":"baseline","horizon":3,"fold":5,"features":392,"volume":2146331000.0,"absolute_error":698271500.0,"signed_error":-40535550.0,"seconds":54.905517}],"direct/thin_offsets/h3_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.330625,"Relative Bias":-0.000121,"|Relative Bias|":0.000121,"метрика конкурса":0.330746,"среднее по дням":0.349659,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":5,"features":236,"volume":69199000000.0,"absolute_error":22878960000.0,"signed_error":-8347499.0,"seconds":23.670605},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.325218,"Relative Bias":-0.014899,"|Relative Bias|":0.014899,"метрика конкурса":0.340118,"среднее по дням":0.361716,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":5,"features":236,"volume":2146331000.0,"absolute_error":698026000.0,"signed_error":-31979150.0,"seconds":23.681039}],"direct/compact_with_additions/h3_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.329533,"Relative Bias":-8.2e-05,"|Relative Bias|":8.2e-05,"метрика конкурса":0.329616,"среднее по дням":0.349218,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":5,"features":249,"volume":69199000000.0,"absolute_error":22803370000.0,"signed_error":-5705204.0,"seconds":23.189139},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32291,"Relative Bias":-0.013385,"|Relative Bias|":0.013385,"метрика конкурса":0.336295,"среднее по дням":0.357733,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":5,"features":249,"volume":2146331000.0,"absolute_error":693072200.0,"signed_error":-28728760.0,"seconds":23.20123}],"direct/baseline/h3_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.330477,"Relative Bias":-7.1e-05,"|Relative Bias|":7.1e-05,"метрика конкурса":0.330547,"среднее по дням":0.352059,"variant":"direct","experiment":"baseline","horizon":3,"fold":6,"features":392,"volume":79671210000.0,"absolute_error":26329490000.0,"signed_error":-5627102.0,"seconds":64.118343},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.332637,"Relative Bias":-0.002148,"|Relative Bias|":0.002148,"метрика конкурса":0.334785,"среднее по дням":0.352283,"variant":"direct","experiment":"baseline","horizon":3,"fold":6,"features":392,"volume":2115532000.0,"absolute_error":703703400.0,"signed_error":-4544340.0,"seconds":64.13321}],"direct/thin_offsets/h3_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.330677,"Relative Bias":-7.8e-05,"|Relative Bias|":7.8e-05,"метрика конкурса":0.330754,"среднее по дням":0.351964,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":6,"features":236,"volume":79671210000.0,"absolute_error":26345400000.0,"signed_error":-6191355.0,"seconds":26.420847},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.333133,"Relative Bias":0.002894,"|Relative Bias|":0.002894,"метрика конкурса":0.336027,"среднее по дням":0.351852,"variant":"direct","experiment":"thin_offsets","horizon":3,"fold":6,"features":236,"volume":2115532000.0,"absolute_error":704753000.0,"signed_error":6121876.0,"seconds":26.431438}],"direct/compact_with_additions/h3_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.329633,"Relative Bias":-6.8e-05,"|Relative Bias|":6.8e-05,"метрика конкурса":0.3297,"среднее по дням":0.351936,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":6,"features":249,"volume":79671210000.0,"absolute_error":26262230000.0,"signed_error":-5396057.0,"seconds":26.960539},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.331696,"Relative Bias":-0.002196,"|Relative Bias|":0.002196,"метрика конкурса":0.333893,"среднее по дням":0.352483,"variant":"direct","experiment":"compact_with_additions","horizon":3,"fold":6,"features":249,"volume":2115532000.0,"absolute_error":701714000.0,"signed_error":-4646208.0,"seconds":26.972029}],"direct/baseline/h4_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333579,"Relative Bias":-0.000558,"|Relative Bias|":0.000558,"метрика конкурса":0.334137,"среднее по дням":0.350729,"variant":"direct","experiment":"baseline","horizon":4,"fold":1,"features":394,"volume":29488060000.0,"absolute_error":9836599000.0,"signed_error":-16463920.0,"seconds":27.221965},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324309,"Relative Bias":-0.000842,"|Relative Bias|":0.000842,"метрика конкурса":0.325151,"среднее по дням":0.351739,"variant":"direct","experiment":"baseline","horizon":4,"fold":1,"features":394,"volume":1942286000.0,"absolute_error":629900000.0,"signed_error":-1636082.0,"seconds":27.238045}],"direct/thin_offsets/h4_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.334435,"Relative Bias":-0.000614,"|Relative Bias|":0.000614,"метрика конкурса":0.335048,"среднее по дням":0.351722,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":1,"features":238,"volume":29488060000.0,"absolute_error":9861834000.0,"signed_error":-18090940.0,"seconds":12.014565},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.324067,"Relative Bias":-0.002009,"|Relative Bias|":0.002009,"метрика конкурса":0.326076,"среднее по дням":0.351771,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":1,"features":238,"volume":1942286000.0,"absolute_error":629430200.0,"signed_error":-3902229.0,"seconds":12.025546}],"direct/compact_with_additions/h4_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.333233,"Relative Bias":-0.000523,"|Relative Bias|":0.000523,"метрика конкурса":0.333756,"среднее по дням":0.353167,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":1,"features":251,"volume":29488060000.0,"absolute_error":9826409000.0,"signed_error":-15418770.0,"seconds":11.548206},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323594,"Relative Bias":-4.6e-05,"|Relative Bias|":4.6e-05,"метрика конкурса":0.323641,"среднее по дням":0.353373,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":1,"features":251,"volume":1942286000.0,"absolute_error":628513100.0,"signed_error":-89680.4,"seconds":11.560219}],"direct/baseline/h4_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332477,"Relative Bias":-0.000183,"|Relative Bias|":0.000183,"метрика конкурса":0.33266,"среднее по дням":0.349236,"variant":"direct","experiment":"baseline","horizon":4,"fold":2,"features":394,"volume":38430030000.0,"absolute_error":12777090000.0,"signed_error":-7039764.0,"seconds":32.941259},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323681,"Relative Bias":-0.003729,"|Relative Bias|":0.003729,"метрика конкурса":0.32741,"среднее по дням":0.367881,"variant":"direct","experiment":"baseline","horizon":4,"fold":2,"features":394,"volume":2089553000.0,"absolute_error":676348100.0,"signed_error":-7792131.0,"seconds":32.955514}],"direct/thin_offsets/h4_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.333198,"Relative Bias":-0.000189,"|Relative Bias|":0.000189,"метрика конкурса":0.333386,"среднее по дням":0.349968,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":2,"features":238,"volume":38430030000.0,"absolute_error":12804800000.0,"signed_error":-7244980.0,"seconds":15.544521},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.323286,"Relative Bias":-0.005009,"|Relative Bias|":0.005009,"метрика конкурса":0.328295,"среднее по дням":0.367181,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":2,"features":238,"volume":2089553000.0,"absolute_error":675523700.0,"signed_error":-10466880.0,"seconds":15.555738}],"direct/compact_with_additions/h4_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.332045,"Relative Bias":-9.2e-05,"|Relative Bias|":9.2e-05,"метрика конкурса":0.332137,"среднее по дням":0.34989,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":2,"features":251,"volume":38430030000.0,"absolute_error":12760520000.0,"signed_error":-3520034.0,"seconds":15.394641},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322881,"Relative Bias":-0.000918,"|Relative Bias|":0.000918,"метрика конкурса":0.3238,"среднее по дням":0.366058,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":2,"features":251,"volume":2089553000.0,"absolute_error":674677300.0,"signed_error":-1919091.0,"seconds":15.405857}],"direct/baseline/h4_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330687,"Relative Bias":-0.000199,"|Relative Bias|":0.000199,"метрика конкурса":0.330886,"среднее по дням":0.347487,"variant":"direct","experiment":"baseline","horizon":4,"fold":3,"features":394,"volume":47967270000.0,"absolute_error":15862160000.0,"signed_error":-9557603.0,"seconds":40.972777},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.327965,"Relative Bias":-0.019218,"|Relative Bias|":0.019218,"метрика конкурса":0.347183,"среднее по дням":0.360601,"variant":"direct","experiment":"baseline","horizon":4,"fold":3,"features":394,"volume":2105647000.0,"absolute_error":690578600.0,"signed_error":-40466010.0,"seconds":40.987689}],"direct/thin_offsets/h4_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.331266,"Relative Bias":-0.000119,"|Relative Bias|":0.000119,"метрика конкурса":0.331385,"среднее по дням":0.348176,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":3,"features":238,"volume":47967270000.0,"absolute_error":15889930000.0,"signed_error":-5706366.0,"seconds":18.579393},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.328111,"Relative Bias":-0.017037,"|Relative Bias|":0.017037,"метрика конкурса":0.345148,"среднее по дням":0.360286,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":3,"features":238,"volume":2105647000.0,"absolute_error":690886100.0,"signed_error":-35873620.0,"seconds":18.59097}],"direct/compact_with_additions/h4_fold3.json":[{"выборка":"train","даты":39,"строки":195000,"WAPE":0.330343,"Relative Bias":-0.000157,"|Relative Bias|":0.000157,"метрика конкурса":0.3305,"среднее по дням":0.348745,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":3,"features":251,"volume":47967270000.0,"absolute_error":15845640000.0,"signed_error":-7538752.0,"seconds":17.969319},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32734,"Relative Bias":-0.015857,"|Relative Bias|":0.015857,"метрика конкурса":0.343196,"среднее по дням":0.356807,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":3,"features":251,"volume":2105647000.0,"absolute_error":689261900.0,"signed_error":-33388370.0,"seconds":17.980823}],"direct/baseline/h4_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.328749,"Relative Bias":-0.000137,"|Relative Bias|":0.000137,"метрика конкурса":0.328886,"среднее по дням":0.345372,"variant":"direct","experiment":"baseline","horizon":4,"fold":4,"features":394,"volume":58105750000.0,"absolute_error":19102200000.0,"signed_error":-7961093.0,"seconds":48.408762},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.330256,"Relative Bias":-0.052122,"|Relative Bias|":0.052122,"метрика конкурса":0.382378,"среднее по дням":0.381932,"variant":"direct","experiment":"baseline","horizon":4,"fold":4,"features":394,"volume":2132154000.0,"absolute_error":704155700.0,"signed_error":-111132800.0,"seconds":48.423709}],"direct/thin_offsets/h4_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.329356,"Relative Bias":-0.000218,"|Relative Bias|":0.000218,"метрика конкурса":0.329574,"среднее по дням":0.346113,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":4,"features":238,"volume":58105750000.0,"absolute_error":19137490000.0,"signed_error":-12638490.0,"seconds":21.87173},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.329959,"Relative Bias":-0.048066,"|Relative Bias|":0.048066,"метрика конкурса":0.378024,"среднее по дням":0.377584,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":4,"features":238,"volume":2132154000.0,"absolute_error":703523100.0,"signed_error":-102483100.0,"seconds":21.884332}],"direct/compact_with_additions/h4_fold4.json":[{"выборка":"train","даты":46,"строки":230000,"WAPE":0.328717,"Relative Bias":-0.000119,"|Relative Bias|":0.000119,"метрика конкурса":0.328836,"среднее по дням":0.34678,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":4,"features":251,"volume":58105750000.0,"absolute_error":19100350000.0,"signed_error":-6916446.0,"seconds":20.624376},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.329311,"Relative Bias":-0.045102,"|Relative Bias|":0.045102,"метрика конкурса":0.374412,"среднее по дням":0.373991,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":4,"features":251,"volume":2132154000.0,"absolute_error":702140600.0,"signed_error":-96164110.0,"seconds":20.635723}],"direct/baseline/h4_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.328278,"Relative Bias":-0.000188,"|Relative Bias|":0.000188,"метрика конкурса":0.328466,"среднее по дням":0.345792,"variant":"direct","experiment":"baseline","horizon":4,"fold":5,"features":394,"volume":68591700000.0,"absolute_error":22517160000.0,"signed_error":-12901090.0,"seconds":57.622364},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.322674,"Relative Bias":-0.027637,"|Relative Bias|":0.027637,"метрика конкурса":0.350311,"среднее по дням":0.362621,"variant":"direct","experiment":"baseline","horizon":4,"fold":5,"features":394,"volume":2188764000.0,"absolute_error":706256800.0,"signed_error":-60490790.0,"seconds":57.636844}],"direct/thin_offsets/h4_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.32879,"Relative Bias":-0.000147,"|Relative Bias|":0.000147,"метрика конкурса":0.328937,"среднее по дням":0.3463,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":5,"features":238,"volume":68591700000.0,"absolute_error":22552240000.0,"signed_error":-10089080.0,"seconds":24.121213},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32303,"Relative Bias":-0.025132,"|Relative Bias|":0.025132,"метрика конкурса":0.348162,"среднее по дням":0.361706,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":5,"features":238,"volume":2188764000.0,"absolute_error":707036700.0,"signed_error":-55008550.0,"seconds":24.132155}],"direct/compact_with_additions/h4_fold5.json":[{"выборка":"train","даты":53,"строки":265000,"WAPE":0.32832,"Relative Bias":-0.000203,"|Relative Bias|":0.000203,"метрика конкурса":0.328523,"среднее по дням":0.346933,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":5,"features":251,"volume":68591700000.0,"absolute_error":22520000000.0,"signed_error":-13920210.0,"seconds":23.863756},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.32207,"Relative Bias":-0.023623,"|Relative Bias|":0.023623,"метрика конкурса":0.345693,"среднее по дням":0.356985,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":5,"features":251,"volume":2188764000.0,"absolute_error":704936100.0,"signed_error":-51704360.0,"seconds":23.875444}],"direct/baseline/h4_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.328558,"Relative Bias":-0.000171,"|Relative Bias|":0.000171,"метрика конкурса":0.328729,"среднее по дням":0.34732,"variant":"direct","experiment":"baseline","horizon":4,"fold":6,"features":394,"volume":79044200000.0,"absolute_error":25970620000.0,"signed_error":-13490470.0,"seconds":66.902665},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.336899,"Relative Bias":-0.001986,"|Relative Bias|":0.001986,"метрика конкурса":0.338885,"среднее по дням":0.355824,"variant":"direct","experiment":"baseline","horizon":4,"fold":6,"features":394,"volume":2123980000.0,"absolute_error":715566100.0,"signed_error":-4219089.0,"seconds":66.917824}],"direct/thin_offsets/h4_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.329045,"Relative Bias":-0.000162,"|Relative Bias|":0.000162,"метрика конкурса":0.329207,"среднее по дням":0.347721,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":6,"features":238,"volume":79044200000.0,"absolute_error":26009110000.0,"signed_error":-12779620.0,"seconds":26.983708},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.336657,"Relative Bias":0.000148,"|Relative Bias|":0.000148,"метрика конкурса":0.336805,"среднее по дням":0.35407,"variant":"direct","experiment":"thin_offsets","horizon":4,"fold":6,"features":238,"volume":2123980000.0,"absolute_error":715053100.0,"signed_error":313635.5,"seconds":26.994465}],"direct/compact_with_additions/h4_fold6.json":[{"выборка":"train","даты":60,"строки":300000,"WAPE":0.328768,"Relative Bias":-0.000126,"|Relative Bias|":0.000126,"метрика конкурса":0.328894,"среднее по дням":0.348498,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":6,"features":251,"volume":79044200000.0,"absolute_error":25987210000.0,"signed_error":-9986167.0,"seconds":27.998198},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.335182,"Relative Bias":-0.003856,"|Relative Bias|":0.003856,"метрика конкурса":0.339038,"среднее по дням":0.354363,"variant":"direct","experiment":"compact_with_additions","horizon":4,"fold":6,"features":251,"volume":2123980000.0,"absolute_error":711919200.0,"signed_error":-8190124.0,"seconds":28.009337}],"direct/baseline/h5_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.325223,"Relative Bias":-0.000401,"|Relative Bias|":0.000401,"метрика конкурса":0.325624,"среднее по дням":0.343281,"variant":"direct","experiment":"baseline","horizon":5,"fold":1,"features":396,"volume":28982240000.0,"absolute_error":9425680000.0,"signed_error":-11620940.0,"seconds":54.759062},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326673,"Relative Bias":-0.002819,"|Relative Bias|":0.002819,"метрика конкурса":0.329492,"среднее по дням":0.339757,"variant":"direct","experiment":"baseline","horizon":5,"fold":1,"features":396,"volume":1895322000.0,"absolute_error":619151200.0,"signed_error":-5342510.0,"seconds":54.774553}],"direct/thin_offsets/h5_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.326751,"Relative Bias":-0.000468,"|Relative Bias|":0.000468,"метрика конкурса":0.327218,"среднее по дням":0.34459,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":1,"features":240,"volume":28982240000.0,"absolute_error":9469967000.0,"signed_error":-13557140.0,"seconds":26.571487},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326877,"Relative Bias":-0.001902,"|Relative Bias|":0.001902,"метрика конкурса":0.328779,"среднее по дням":0.338596,"variant":"direct","experiment":"thin_offsets","horizon":5,"fold":1,"features":240,"volume":1895322000.0,"absolute_error":619537200.0,"signed_error":-3604476.0,"seconds":26.58236}],"direct/compact_with_additions/h5_fold1.json":[{"выборка":"train","даты":25,"строки":125000,"WAPE":0.32647,"Relative Bias":-0.000384,"|Relative Bias|":0.000384,"метрика конкурса":0.326854,"среднее по дням":0.346847,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":1,"features":253,"volume":28982240000.0,"absolute_error":9461838000.0,"signed_error":-11118120.0,"seconds":27.468946},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.326679,"Relative Bias":-0.003152,"|Relative Bias|":0.003152,"метрика конкурса":0.329831,"среднее по дням":0.342278,"variant":"direct","experiment":"compact_with_additions","horizon":5,"fold":1,"features":253,"volume":1895322000.0,"absolute_error":619162000.0,"signed_error":-5974875.0,"seconds":27.481195}],"direct/baseline/h5_fold2.json":[{"выборка":"train","даты":32,"строки":160000,"WAPE":0.324645,"Relative Bias":-0.000327,"|Relative Bias|":0.000327,"метрика конкурса":0.324971,"среднее по дням":0.342208,"variant":"direct","experiment":"baseline","horizon":5,"fold":2,"features":396,"volume":37770830000.0,"absolute_error":12262090000.0,"signed_error":-12335590.0,"seconds":68.901301},{"выборка":"tuning","даты":7,"строки":7000,"WAPE":0.327693,"Relative Bias":0.004008,"|Relative Bias|":0.004008,"метрика конкурса":0.331701,"среднее по дням":0.353419,"variant":"direct","experiment":"baseline","horizon":5,"fold":2,"features":396,"volume":2037994000.0,"absolute_error":667837000.0,"signed_error":8168032.0,"seconds":68.917342}]}''')
for relative_path, rows in saved_comparisons.items():
    comparison_path = RUN_DIR / 'comparisons' / relative_path
    if not comparison_path.exists():
        comparison_path.parent.mkdir(parents=True, exist_ok=True)
        atomic_json(comparison_path, rows)


## Сравнение признаков и подбор выбранных моделей

In [ ]:
import re

EXPERIMENTS = {
    'baseline': dict(),
    'drop_constant': dict(drop_constant=True),
    'compact_availability': dict(compact_availability=True),
    'calendar_dayofweek': dict(calendar=True),
    'thin_offsets': dict(thin=True),
    'drop_network': dict(network='drop'),
    'thin_network': dict(network='thin'),
    'add_time': dict(add=['time']),
    'add_profile': dict(add=['profile']),
    'add_level24': dict(add=['level']),
    'add_anomaly': dict(add=['anomaly']),
    'add_trends': dict(add=['trend']),
    'add_volatility': dict(add=['volatility']),
    'add_seasonal_dispersion': dict(add=['dispersion']),
    'all_additions': dict(add=['time','profile','level','anomaly','trend','volatility','dispersion']),
    'compact': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin'),
    'compact_with_additions': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin',
                                   add=['time','profile','level','anomaly','trend','volatility','dispersion'])
}


def experiment_columns(names, spec):
    result = []
    for name in names:
        if name.startswith('x_'):
            if any(name.startswith(f'x_{group}_') for group in spec.get('add', [])):
                result.append(name)
            continue
        if spec.get('drop_constant') and name == 'selected_status6_pair_count':
            continue
        if spec.get('compact_availability') and 'signal_available' in name:
            continue
        if spec.get('calendar') and name in ('weekday_sin','weekday_cos','is_weekend'):
            continue
        if spec.get('network') == 'drop' and name.startswith('network_'):
            continue
        if spec.get('network') == 'thin' and name.startswith('network_') and '_offset_' in name:
            continue
        if spec.get('thin') and not name.startswith('slot_') and re.search(r'_offset_m[134]$', name):
            continue
        result.append(name)
    if spec.get('compact_availability'):
        for name in ('status6_lag_valid', 'residual_status6_lag_valid'):
            if name in names and name not in result:
                result.append(name)
    return result


base_load_stage = load_stage


def load_stage(folder):
    data = base_load_stage(folder)
    metadata = json.loads((folder / 'ready.json').read_text())
    names = list(joblib.load(folder / 'preprocess.joblib').feature_names_in_)
    selected_names = experiment_columns(names, EXPERIMENTS[ACTIVE_EXPERIMENT])
    indices = [names.index(name) for name in selected_names]
    return {name: (frame, matrix[:, indices]) for name, (frame, matrix) in data.items()}


def select_context(variant, experiment):
    global RESIDUAL, ACTIVE_EXPERIMENT, DOCS_DIR, PARAMS_DIR, POSTPROCESS_DIR, MODEL_DIR
    RESIDUAL = variant == 'residual'
    ACTIVE_EXPERIMENT = experiment
    folder = RUN_DIR / variant
    DOCS_DIR = folder / 'docs'
    PARAMS_DIR = folder / 'model_params'
    POSTPROCESS_DIR = folder / 'postprocessing'
    MODEL_DIR = folder / 'models'
    for directory in (DOCS_DIR, PARAMS_DIR, POSTPROCESS_DIR, MODEL_DIR):
        directory.mkdir(parents=True, exist_ok=True)


def comparison_fit(variant, experiment, h, i):
    select_context(variant, experiment)
    path = RUN_DIR / 'comparisons' / variant / experiment / f'h{h}_fold{i+1}.json'
    if path.exists():
        rows = json.loads(path.read_text())
        display(pd.DataFrame(rows))
        return rows
    fold = folds[i]
    started = time.perf_counter()
    folder = prepare_stage(h, f'fold{i+1}', fold['train'], fold['tuning'], fold['val'][0].normalize())
    data = load_stage(folder)
    train, matrix = data['train']
    model = fit_forest(train, matrix, base_params_for_horizon(h))
    rows = []
    for key, split in [('train','train'),('valid','tuning')]:
        frame, x = data[key]
        prediction = predict_matrix(model, frame, x)
        row = score_row(frame, prediction, split)
        error = prediction - frame.y.to_numpy()
        row.update(variant=variant, experiment=experiment, horizon=h, fold=i+1, features=x.shape[1],
                   volume=float(frame.y.sum()), absolute_error=float(np.abs(error).sum()), signed_error=float(error.sum()),
                   seconds=time.perf_counter()-started)
        rows.append(row)
    path.parent.mkdir(parents=True, exist_ok=True)
    atomic_json(path, rows)
    display(pd.DataFrame(rows))
    del model, data, train, matrix, frame, x
    gc.collect()
    return rows


unknown = set(EXPERIMENT_NAMES) - set(EXPERIMENTS)
if unknown or 'baseline' not in EXPERIMENT_NAMES:
    raise ValueError(f'Неверный список экспериментов: {unknown}; baseline обязателен')
if not VARIANTS or set(VARIANTS) - {'direct','residual'}:
    raise ValueError('VARIANTS должен содержать direct и/или residual')
comparison_fits = len(VARIANTS) * len(EXPERIMENT_NAMES) * len(HORIZONS) * len(folds)
print(f'Сравнение: {comparison_fits} обучений; Optuna: {len(VARIANTS)*len(HORIZONS)*N_TRIALS*len(folds)}; test: {len(VARIANTS)*len(HORIZONS)}')

comparison_records = []
for variant in VARIANTS:
    for h in HORIZONS:
        for i in range(len(folds)):
            for experiment in EXPERIMENT_NAMES:
                comparison_records.extend(comparison_fit(variant, experiment, h, i))
comparison = pd.DataFrame(comparison_records)
comparison.to_csv(RUN_DIR / 'feature_experiments_by_horizon.csv', index=False)
pooled = comparison.groupby(['variant','experiment','fold','выборка'], as_index=False)[['volume','absolute_error','signed_error']].sum()
pooled['WAPE'] = pooled.absolute_error / pooled.volume
pooled['Relative Bias'] = pooled.signed_error / pooled.volume
pooled['|Relative Bias|'] = pooled['Relative Bias'].abs()
pooled['метрика конкурса'] = pooled.WAPE + pooled['|Relative Bias|']
pooled.to_csv(RUN_DIR / 'feature_experiments_by_fold.csv', index=False)
summary = pooled.loc[pooled['выборка'].eq('tuning')].groupby(['variant','experiment'],as_index=False)[['WAPE','Relative Bias','|Relative Bias|','метрика конкурса']].mean()
baseline = summary.loc[summary.experiment.eq('baseline'),['variant','метрика конкурса']].rename(columns={'метрика конкурса':'baseline_score'})
summary = summary.merge(baseline,on='variant')
summary['delta_vs_baseline'] = summary['метрика конкурса'] - summary.baseline_score
summary = summary.sort_values(['variant','метрика конкурса','experiment'])
summary.to_csv(RUN_DIR / 'feature_experiments_summary.csv',index=False)
display(summary)
tuning_details = comparison.loc[comparison['выборка'].eq('tuning')].copy()
reference = tuning_details.loc[tuning_details.experiment.eq('baseline'), ['variant','horizon','fold','метрика конкурса']].rename(columns={'метрика конкурса':'baseline_score'})
deltas = tuning_details.merge(reference, on=['variant','horizon','fold'])
deltas['delta_vs_baseline'] = deltas['метрика конкурса'] - deltas.baseline_score
deltas.to_csv(RUN_DIR / 'feature_deltas_by_window.csv', index=False)
stability = deltas.groupby(['variant','experiment','horizon'], as_index=False).agg(
    mean_score=('метрика конкурса','mean'), worst_score=('метрика конкурса','max'),
    score_std=('метрика конкурса','std'), mean_delta=('delta_vs_baseline','mean'),
    wins=('delta_vs_baseline', lambda values: int((values < 0).sum())), windows=('fold','count'),
    mean_wape=('WAPE','mean'), mean_signed_bias=('Relative Bias','mean'), mean_abs_bias=('|Relative Bias|','mean'))
stability.to_csv(RUN_DIR / 'feature_stability_by_horizon.csv', index=False)
display(stability)
winners = {variant: summary.loc[summary.variant.eq(variant)].iloc[0].experiment for variant in VARIANTS}
atomic_json(RUN_DIR / 'feature_selection_frozen.json', dict(winners=winners, selection_end_exclusive=str(selection_end), base_params=BASE_PARAMS_BY_HORIZON))

all_selected = {}
for variant in VARIANTS:
    select_context(variant, winners[variant])
    selected = {}
    for h in HORIZONS:
        selected[h] = select_horizon(h)
    all_selected[variant] = selected
    atomic_json(PARAMS_DIR / 'selection_frozen.json', selected)
atomic_json(RUN_DIR / 'all_parameters_frozen.json', all_selected)

parameter_comparisons = []
for variant in VARIANTS:
    select_context(variant, winners[variant])
    for h in HORIZONS:
        study = optuna.load_study(study_name=f'h{h}', storage=f'sqlite:///{(POSTPROCESS_DIR / "search.sqlite3").resolve().as_posix()}')
        completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
        base_trials = [trial for trial in completed if trial.params == base_params_for_horizon(h)]
        if not base_trials:
            raise ValueError('Исходная конфигурация не проверена в Optuna')
        base_trial = min(base_trials, key=lambda trial: trial.value)
        best_trial = study.best_trial
        if best_trial.value > base_trial.value + 1e-12:
            raise ValueError('Подбор проиграл исходной конфигурации')
        for label, trial in [('base',base_trial),('optuna',best_trial)]:
            for i in range(len(folds)):
                rows = json.loads((DOCS_DIR / f'search_h{h}_trial{trial.number}_fold{i+1}.json').read_text())
                for row in rows:
                    parameter_comparisons.append(dict(variant=variant, configuration=label, experiment=winners[variant],
                                                     params=json.dumps(trial.params, sort_keys=True), **row))
    details = pd.DataFrame([row for row in parameter_comparisons if row['variant'] == variant])
    details.to_csv(DOCS_DIR / 'base_vs_optuna_by_fold.csv', index=False)
    aggregate = details.loc[details['выборка'].eq('tuning')].groupby(['horizon','configuration'],as_index=False).agg(
        WAPE=('WAPE','mean'), signed_bias=('Relative Bias','mean'), absolute_bias=('|Relative Bias|','mean'),
        score=('метрика конкурса','mean'), worst_window=('метрика конкурса','max'))
    base_score = aggregate.loc[aggregate.configuration.eq('base'),['horizon','score']].rename(columns={'score':'base_score'})
    aggregate = aggregate.merge(base_score,on='horizon')
    aggregate['delta_vs_base'] = aggregate.score - aggregate.base_score
    aggregate.to_csv(DOCS_DIR / 'base_vs_optuna_summary.csv', index=False)
    display(aggregate)


## Независимая оценка после фиксации выбора

In [ ]:
for variant in VARIANTS:
    select_context(variant, winners[variant])
    selected = all_selected[variant]
    manifest = dict(variant=variant, experiment=winners[variant], contract=RUN_CONTRACT,
                    test_note='Независимая оценка test получена после фиксации признаков и параметров, до какого-либо переобучения на всей истории. Переобучение на всей истории и submission не выполняются. Период test ранее использовался в проекте и не является новым слепым test.')
    atomic_json(DOCS_DIR / 'manifest.json', manifest)
    evaluation_rows = []
    for h in HORIZONS:
        checkpoint = MODEL_DIR / 'evaluation' / f'h{h}'
        done = checkpoint / 'complete.json'
        if done.exists():
            rows = json.loads(done.read_text())['rows']
            display(pd.DataFrame(rows))
        else:
            started = time.perf_counter()
            folder = prepare_stage(h, 'evaluation', validation_dates=folds[-1]['test'], boundary=holdout_start.normalize())
            data = load_stage(folder)
            train, x_train = data['train']
            model = fit_forest(train, x_train, selected[h]['params'])
            rows = []
            for name, split in [('train', 'train'), ('valid', 'test')]:
                frame, matrix = data[name]
                prediction = predict_matrix(model, frame, matrix)
                rows.append(score_row(frame, prediction, split))
                part = prediction_part(frame, prediction, h, split)
                part['model_info_through'] = holdout_start.normalize() - STEP
                part['is_unbiased_oof'] = split == 'test'
                path = POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet'
                temporary = path.with_suffix('.partial')
                part.to_parquet(temporary, index=False)
                temporary.replace(path)
            checkpoint.mkdir(parents=True, exist_ok=True)
            metadata = json.loads((folder / 'ready.json').read_text())
            encoded_names = list(joblib.load(folder / 'preprocess.joblib').feature_names_in_)
            selected_names = experiment_columns(encoded_names, EXPERIMENTS[ACTIVE_EXPERIMENT])
            metadata.update(experiment=ACTIVE_EXPERIMENT, selected_features=selected_names, selected_indices=[encoded_names.index(n) for n in selected_names])
            metadata.update(algorithm=ALGORITHM, residual=RESIDUAL, target_scale=TARGET_SCALE, route_categories=routes.tolist(), route_encoding='binary_index_plus_one', params=selected[h]['params'])
            bundle = dict(model=model, preprocess=joblib.load(folder / 'preprocess.joblib'), metadata=metadata)
            temporary = checkpoint / 'model.partial'
            joblib.dump(bundle, temporary, compress=0)
            temporary.replace(checkpoint / 'model.joblib')
            atomic_json(checkpoint / 'metadata.json', metadata)
            report(rows, 'independent_evaluation', h, seconds=time.perf_counter() - started)
            atomic_json(done, dict(rows=rows))
            del bundle, model, data, train, x_train, frame, matrix, part
            gc.collect()
        evaluation_rows.extend([dict(horizon=h, **row) for row in rows])
    pd.DataFrame(evaluation_rows).to_csv(DOCS_DIR / 'independent_metrics_by_horizon.csv', index=False)
    aggregate_rows = []
    for split in ('train', 'test'):
        part = pd.concat([pd.read_parquet(POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet') for h in HORIZONS], ignore_index=True)
        aggregate_rows.append(score_row(part, part.prediction, split))
        aggregate_rows.append(score_row(part, part.profile28, split + ' profile28'))
        if split == 'test':
            part.to_parquet(POSTPROCESS_DIR / 'test_predictions.parquet', index=False)
        del part
    independent_metrics = pd.DataFrame(aggregate_rows)
    display(independent_metrics)
    independent_metrics.to_csv(DOCS_DIR / 'independent_metrics.csv', index=False)
    (DOCS_DIR / 'evaluation_note.txt').write_text(manifest['test_note'])
    records = []
    for path in sorted(DOCS_DIR.glob('*_h*_trial*_fold*.json')):
        records.extend(json.loads(path.read_text()))
    pd.DataFrame(records).to_csv(DOCS_DIR / 'metrics.csv', index=False)
    with zipfile.ZipFile(RUN_DIR / variant / 'reports.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in (DOCS_DIR, PARAMS_DIR):
            for path in folder.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(RUN_DIR / variant))
    print(manifest['test_note'])
    print(f'Результаты: {RUN_DIR.resolve()}')